# Downstream Limit-Order Benchmark

Single-shot limit-order placement task: sell a fixed quantity $Q$ via one limit-sell order at arrival, with a deadline of $H = 20$ snapshots (100 s). The agent picks an offset $d \in \{-1, 0, 1, 2, 3, 4, 5\}$ where $-1$ means immediate market sell, $0$ means post at touch, and $d \ge 1$ means $d$ ticks above mid.

Compare:
- **Static offsets** $\delta_{-1}$ (Market) through $\delta_{+5}$: each fixed offset evaluated as its own strategy.
- **LOBGrad** (`JointTimeGrad`): per-window choice using LOBGrad rollouts.
- **Oracle**: per-window choice using the realized future as rollout (ceiling).

Metric: realized Implementation Shortfall per share = arrival_mid − avg_fill_price (lower is better). We report mean IS and CVaR$_{95}$(IS) across 500 windows.

## §0  Configuration

In [1]:
# ─── Models to include ────────────────────────────────────────────────────
ACTIVE_MODELS = ["JointTimeGrad"]
INCLUDE_BOOTSTRAP = False

# ─── Checkpoints ──────────────────────────────────────────────────────────
TG_CKPT     = "checkpoints/final/joint_timegrad/best_model.pt"
TG_LABEL    = "JointTimeGrad"

# ─── Architecture (matches the trial_88 final model) ─────────────────────
K              = 5         # LOB levels per side
PRICE_LEVEL    = 2.0       # tick size δ in USDT
CONTEXT_LEN    = 100       # conditioning window length
HORIZON        = 20        # generation horizon
N_WINDOWS      = 500       # evaluation windows

EMB_DIM        = 32
HIDDEN_ENC     = 151
HIDDEN_NOISE   = 255
DIFFUSION_K    = 102
NUM_ENC_LAYERS = 1
CELL_TYPE      = "LSTM"
USE_COND_PROJ  = True

# ─── Limit-order benchmark settings ──────────────────────────────────────
OFFSET_GRID  = [-1, 0, 1, 2, 3, 4, 5]   # static rungs + rollout-based candidates
N_ROLLOUTS   = 100                      # rollouts per window per model
OBJECTIVE    = "mean"                   # rollout-based selection: minimize mean IS
ALPHA        = 0.95                     # CVaR tail level
FILL_RATIO   = 0.8                      # depth-cleared trigger threshold

# ─── Output ──────────────────────────────────────────────────────────────
SAVE_FIGS  = True
TABLE_DIR  = "figures/tables"
FIG_DIR    = "figures/main"

## §1  Imports

In [ ]:
import os, sys
from pathlib import Path

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

REPO_ROOT = Path.cwd()
while REPO_ROOT.name == "notebooks" or not (REPO_ROOT / "lobgrad").is_dir():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("could not locate repo root containing lobgrad/")
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"REPO_ROOT = {REPO_ROOT}")
os.chdir(REPO_ROOT)

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 110

from lobgrad.config import BenchmarkConfig
from lobgrad.data.splits import date_splits_joint, sample_eval_windows_joint
from lobgrad.execution.benchmark import ExecutionWindow, calibrate_q_from_depth
from lobgrad.execution.rollouts  import ModelRollouts, BootstrapRollouts
from lobgrad.execution.limit_order import (
    simulate_limit_sell, run_limit_order_benchmark, cvar_upper,
)

## §2  Data — splits and evaluation windows

In [ ]:
cfg = BenchmarkConfig(
    levels                       = K,
    price_level                  = PRICE_LEVEL,
    queue_repr                   = "signed",
    queue_mode                   = "absolute",
    timegrad_checkpoint          = TG_CKPT,
    timegrad_train               = False,
    timegrad_k                   = DIFFUSION_K,
    timegrad_hidden_enc          = HIDDEN_ENC,
    timegrad_hidden_noise        = HIDDEN_NOISE,
    timegrad_num_enc_layers      = NUM_ENC_LAYERS,
    timegrad_emb_dim             = EMB_DIM,
    timegrad_cell_type           = CELL_TYPE,
    timegrad_use_cond_projection = USE_COND_PROJ,
    timegrad_loss_type           = "l2",
    timegrad_schedule            = "linear",
    timegrad_beta_start          = 0.0001,
    timegrad_beta_end            = 0.1,
)

(ts_train, X_train, anchors_train), \
(ts_val,   X_val,   anchors_val),   \
(ts_test,  X_test,  anchors_test),  \
normalizer = date_splits_joint(cfg)

print(f"Train: {len(X_train):>7,d}  Val: {len(X_val):>5,d}  Test: {len(X_test):>5,d}")

rng_windows = np.random.default_rng(2) # Use different random seed to paper_figures.ipynb for independent experiment
test_ctx_q, test_fut_q, test_ctx_a, test_fut_a = sample_eval_windows_joint(
    X_test, anchors_test, CONTEXT_LEN, HORIZON, N_WINDOWS, rng=rng_windows,
)
print(f"\nWindows  ctx_q={test_ctx_q.shape}  fut_q={test_fut_q.shape}")
print(f"         ctx_a={test_ctx_a.shape}  fut_a={test_fut_a.shape}")

test_fut_q_raw = np.abs(np.stack([
    normalizer.inverse_transform(test_fut_q[i]) for i in range(N_WINDOWS)
]))
print(f"         fut_q_raw={test_fut_q_raw.shape}  range=[{test_fut_q_raw.min():.3f}, {test_fut_q_raw.max():.3f}]")

Train: 223,378  Val: 28,530  Test: 56,918

Windows  ctx_q=(500, 100, 10)  fut_q=(500, 20, 10)
         ctx_a=(500, 100)  fut_a=(500, 20)
         fut_q_raw=(500, 20, 10)  range=[0.000, 810.212]


## §3  Load models

In [4]:
from lobgrad.execution.simulate import JointTimeGradGenerator


GENERATORS = {}

if TG_LABEL in ACTIVE_MODELS:
    print(f"Loading {TG_LABEL} from {TG_CKPT}")
    GENERATORS[TG_LABEL] = JointTimeGradGenerator(cfg, normalizer)

print(f"\n=> {len(GENERATORS)} generators loaded: {list(GENERATORS)}")

Loading JointTimeGrad from checkpoints/final/joint_timegrad/best_model.pt
[JointTimeGrad] Loaded checkpoint from checkpoints/final/joint_timegrad/best_model.pt

=> 1 generators loaded: ['JointTimeGrad']


## §4  Build rollout sources and execution windows

`Q` is calibrated from the training-set bid-side top-3 depth.

In [5]:
# Convert training queue to raw absolute sizes (one-time op, also used by Bootstrap).
train_queue_raw = np.abs(normalizer.inverse_transform(X_train))

# Calibrate Q from training-set bid-side top-3 depth (we are SELLING).
Q_TOTAL = calibrate_q_from_depth(
    train_queue_raw = train_queue_raw,
    levels          = K,
    side            = "sell",
    levels_to_avg   = 3,
    multiplier      = 2.0,
)
print(f"Calibrated Q_TOTAL = {Q_TOTAL:.2f}  (1.0 × top-3 bid depth)")

SOURCES = {}
if INCLUDE_BOOTSTRAP:
    SOURCES["Bootstrap"] = BootstrapRollouts(
        train_queue_norm = X_train,
        train_anchors    = anchors_train,
        normalizer       = normalizer,
        rng              = np.random.default_rng(1),
    )
for label, gen in GENERATORS.items():
    SOURCES[label] = ModelRollouts(
        generator   = gen,
        normalizer  = normalizer,
        context_len = CONTEXT_LEN,
        label       = label,
    )
print("\nRollout sources:")
for name in SOURCES:
    print(f"  {name}  ({type(SOURCES[name]).__name__})")


def make_window(i):
    return ExecutionWindow(
        real_mid          = test_fut_a[i].astype(np.float64),
        real_queue_raw    = test_fut_q_raw[i].astype(np.float64),
        real_queue_norm   = test_fut_q[i].astype(np.float64),
        prefix_anchors    = test_ctx_a[i].astype(np.float64),
        prefix_queue_norm = test_ctx_q[i].astype(np.float64),
    )

benchmark_windows = [make_window(i) for i in range(N_WINDOWS)]
print(f"\nBuilt {len(benchmark_windows)} execution windows")

Calibrated Q_TOTAL = 29.38  (1.0 × top-3 bid depth)

Rollout sources:
  JointTimeGrad  (ModelRollouts)

Built 500 execution windows


## §5  Run benchmark

Two passes:
1. **LOBGrad + Oracle** — `run_limit_order_benchmark` does the rollout-and-pick step per window.
2. **All static offsets** — every $\delta_j \in \{-1, 0, 1, \ldots, 5\}$ as its own fixed-offset strategy.

These are merged into a single results dict, then summarized.

In [6]:
# Pass 1: model + bootstrap + oracle (with auto Market/Touch/Passive baselines).
results = run_limit_order_benchmark(
    windows      = benchmark_windows,
    sources      = SOURCES,
    Q            = Q_TOTAL,
    offset_grid  = OFFSET_GRID,
    n_rollouts   = N_ROLLOUTS,
    price_level  = PRICE_LEVEL,
    levels       = K,
    objective    = OBJECTIVE,
    alpha        = ALPHA,
    fill_ratio   = FILL_RATIO,
    add_static   = True,
    add_oracle   = True,
    progress     = True,
)
print(f"\nPass 1 done: sources={list(results)}")

Limit-order benchmark: 100%|██████████| 500/500 [03:20<00:00,  2.49it/s]


Pass 1 done: sources=['Market', 'Touch', 'Passive', 'Oracle', 'JointTimeGrad']


In [ ]:
# Pass 2: every static offset on the realized future. Drop the auto-added
# Market/Touch/Passive labels so we can replace them with explicit j-labels.
for redundant in ("Market", "Touch", "Passive"):
    results.pop(redundant, None)


def offset_label(d: int) -> str:
    if d == -1:
        return r"$j={-1}$ (Market)"
    return rf"$j={{{d}}}$"


for d in OFFSET_GRID:
    is_arr, status_arr = [], []
    for win in benchmark_windows:
        outcome = simulate_limit_sell(
            Q             = Q_TOTAL,
            offset_level  = d,
            arrival_mid   = float(win.real_mid[0]),
            arrival_queue = win.real_queue_raw[0],
            future_mid    = win.real_mid,
            future_queue  = win.real_queue_raw,
            price_level   = PRICE_LEVEL,
            levels        = K,
            fill_ratio    = FILL_RATIO,
        )
        is_arr.append(outcome.is_per_share)
        status_arr.append(outcome.status)
    results[offset_label(d)] = {
        "selected_offset": np.full(len(benchmark_windows), d, dtype=np.int32),
        "realized_is":     np.asarray(is_arr,     dtype=np.float64),
        "status":          np.asarray(status_arr, dtype=object),
    }

print(f"Pass 2 done: total sources = {len(results)}")

## §6  Summary table — mean IS and CVaR$_{95}$(IS) with 95% bootstrap CIs

Bootstrap is over the 500 test windows (resampling indices with replacement, $B=10{,}000$). This captures window-sampling variance only; rollout noise from `JointTimeGrad` is not reflected.

In [8]:
# Compute mean IS and CVaR_95(IS) per source, with 95% bootstrap CIs
# over the 500 test windows.
N_BOOT     = 10_000
CI_ALPHA   = 0.05  # 95% CI
boot_rng   = np.random.default_rng(0)


def bootstrap_ci(stat_fn, x, n_boot=N_BOOT, alpha=CI_ALPHA, rng=None):
    """Percentile bootstrap CI for stat_fn applied to resamples of x."""
    rng  = boot_rng if rng is None else rng
    n    = len(x)
    idx  = rng.integers(0, n, size=(n_boot, n))
    boot = np.fromiter(
        (stat_fn(x[row]) for row in idx),
        dtype=np.float64, count=n_boot,
    )
    lo = float(np.percentile(boot, 100 * alpha / 2))
    hi = float(np.percentile(boot, 100 * (1 - alpha / 2)))
    return lo, hi


rows = []
for name, d in results.items():
    is_arr = d["realized_is"]
    mean_lo, mean_hi = bootstrap_ci(np.mean, is_arr)
    cvar_lo, cvar_hi = bootstrap_ci(lambda v: cvar_upper(v, ALPHA), is_arr)
    rows.append({
        "source":  name,
        "mean_is": float(is_arr.mean()),
        "mean_lo": mean_lo,
        "mean_hi": mean_hi,
        "cvar_is": cvar_upper(is_arr, ALPHA),
        "cvar_lo": cvar_lo,
        "cvar_hi": cvar_hi,
    })
summary = pd.DataFrame(rows).set_index("source")

# Order rows: static rungs (in offset order), then learned models, then Oracle.
static_order  = [offset_label(d) for d in OFFSET_GRID]
learned_order = [s for s in ("Bootstrap", "CGAN", "JointTimeGrad") if s in summary.index]
row_order     = static_order + learned_order + ["Oracle"]
summary       = summary.reindex([r for r in row_order if r in summary.index])


def fmt_ci(point, lo, hi):
    return f"{point:6.2f}  [{lo:6.2f}, {hi:6.2f}]"


display_df = pd.DataFrame({
    "Mean IS [95% CI]":      [fmt_ci(r.mean_is, r.mean_lo, r.mean_hi) for r in summary.itertuples()],
    "CVaR_95(IS) [95% CI]":  [fmt_ci(r.cvar_is, r.cvar_lo, r.cvar_hi) for r in summary.itertuples()],
}, index=summary.index)

print("\n=== Limit-order benchmark — IS per share (USDT), 95% bootstrap CI ===\n")
print(display_df.to_string())


=== Limit-order benchmark — IS per share (USDT), 95% bootstrap CI ===

                              Mean IS [95% CI]      CVaR_95(IS) [95% CI]
source                                                                  
Market                  5.89  [  5.74,   6.04]    8.45  [  8.34,   8.53]
$\delta_{0}$ (Touch)    1.75  [  0.94,   2.65]   38.90  [ 29.43,  47.52]
$\delta_{+1}$           0.75  [ -0.18,   1.78]   43.42  [ 34.11,  51.49]
$\delta_{+2}$           0.56  [ -0.61,   1.77]   47.86  [ 38.37,  56.99]
$\delta_{+3}$           2.93  [  1.60,   4.33]   51.51  [ 42.56,  59.76]
$\delta_{+4}$           3.86  [  2.45,   5.30]   51.33  [ 43.09,  59.24]
$\delta_{+5}$           4.56  [  3.06,   6.04]   52.65  [ 44.66,  60.70]
JointTimeGrad           0.03  [ -1.03,   1.18]   43.22  [ 35.39,  50.13]
Oracle                 -6.73  [ -7.14,  -6.32]    7.07  [  2.99,   7.74]


## §7  LaTeX table

In [ ]:
def to_latex_table(df):
    BS, EOL = chr(92), " " + chr(92) + chr(92)
    candidates = [r for r in df.index if r != "Oracle"]
    best_mean  = df.loc[candidates, "mean_is"].idxmin()
    best_cvar  = df.loc[candidates, "cvar_is"].idxmin()

    # Display-name remapping (internal source name -> table row label).
    DISPLAY = {"JointTimeGrad": "LOBGrad-optimized"}

    def cell(r, point_col, lo_col, hi_col, best):
        p_str  = f"{df.loc[r, point_col]:.2f}"
        lo_str = f"{df.loc[r, lo_col]:.2f}"
        hi_str = f"{df.loc[r, hi_col]:.2f}"
        main = (BS + "textbf{" + p_str + "}") if r == best else p_str
        ci_line = ("{" + BS + "scriptsize $["
                   + lo_str + "," + BS + "," + hi_str + "]$}")
        return BS + "shortstack[r]{" + main + " " + BS + BS + " " + ci_line + "}"

    body = []
    for r in df.index:
        if r == "Oracle":
            body.append(BS + "midrule")
        body.append(" & ".join([
            DISPLAY.get(r, r),
            cell(r, "mean_is", "mean_lo", "mean_hi", best_mean),
            cell(r, "cvar_is", "cvar_lo", "cvar_hi", best_cvar),
        ]) + EOL)

    cap = (
        f"Limit-order execution benchmark across {len(benchmark_windows)} "
        f"test windows. IS per share in USDT (lower is better). "
        f"Sub-rows give 95\\% bootstrap CIs over windows ($B={N_BOOT:,}$). "
        + BS + "textbf{Bold}: best non-Oracle row per column."
    )

    return "\n".join([
        BS + "begin{table}[t]",
        BS + "caption{" + cap + "}",
        BS + "label{tab:execution_limit_order}",
        BS + "centering", BS + "small",
        BS + "renewcommand{" + BS + "arraystretch}{1.25}",
        BS + "setlength{" + BS + "tabcolsep}{4pt}",
        BS + "begin{tabular}{lrr}",
        BS + "toprule",
        "Strategy & Mean IS & $" + BS + "mathrm{CVaR}_{95}(" + BS + "mathrm{IS})$ " + EOL,
        BS + "midrule",
        *body,
        BS + "bottomrule", BS + "end{tabular}",
        BS + "end{table}",
    ])


tex = to_latex_table(summary)
print(tex)

if SAVE_FIGS:
    os.makedirs(TABLE_DIR, exist_ok=True)
    csv_path = os.path.join(TABLE_DIR, "table_execution_limit_order.csv")
    tex_path = os.path.join(TABLE_DIR, "table_execution_limit_order.tex")
    summary.to_csv(csv_path)
    with open(tex_path, "w") as f:
        f.write(tex)
    print(f"\nsaved {csv_path}")
    print(f"saved {tex_path}")